In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import sys

repo_root = os.path.abspath("..")

if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.utils.common import (
    read_config,
    build_source_path,
    build_table_name,
    read_json_file,
    read_csv_file,
    read_excel_file,
    standardize_column_names,
    add_ingestion_columns,
    write_delta_table
)

In [0]:
config_path = os.path.join(repo_root, "config", "dev_config.json")

config = read_config(config_path)

config

In [0]:
orders_path = build_source_path(config, "orders")
customers_path = build_source_path(config, "customers")
products_path = build_source_path(config, "products")

orders_file_name = config["source"]["files"]["orders"]["file_name"]
customers_file_name = config["source"]["files"]["customers"]["file_name"]
products_file_name = config["source"]["files"]["products"]["file_name"]

print("Orders path   :", orders_path)
print("Customers path:", customers_path)
print("Products path :", products_path)

In [0]:
raw_orders_table = build_table_name(config, "raw", "orders")
raw_customers_table = build_table_name(config, "raw", "customers")
raw_products_table = build_table_name(config, "raw", "products")

print("Raw orders table   :", raw_orders_table)
print("Raw customers table:", raw_customers_table)
print("Raw products table :", raw_products_table)

In [0]:
try:
    orders_df = read_json_file(spark, orders_path)
    customers_df = read_excel_file(spark, customers_path)
    products_df = read_csv_file(spark, products_path)

    print("Source files read successfully.")

except Exception as error:
    raise RuntimeError(
        "Failed while reading one or more source files. "
        "Please verify source file paths, file formats, and required Python packages."
    ) from error

In [0]:
try:
    orders_df = (
        orders_df
        .transform(standardize_column_names)
        .transform(lambda df: add_ingestion_columns(df, orders_file_name))
    )

    customers_df = (
        customers_df
        .transform(standardize_column_names)
        .transform(lambda df: add_ingestion_columns(df, customers_file_name))
    )

    products_df = (
        products_df
        .transform(standardize_column_names)
        .transform(lambda df: add_ingestion_columns(df, products_file_name))
    )

    print("Column standardization and ingestion metadata completed successfully.")

except Exception as error:
    raise RuntimeError("Failed while standardizing columns or adding ingestion metadata.") from error

In [0]:
try:
    write_delta_table(orders_df, raw_orders_table)
    write_delta_table(customers_df, raw_customers_table)
    write_delta_table(products_df, raw_products_table)

    print("Raw tables loaded successfully.")

except Exception as error:
    raise RuntimeError("Failed while writing data to raw Delta tables.") from error